# U9a · Modelos fundacionales — usar Hugging Face con `pipeline()`

> **Para quien pone el criterio clínico, no el código.** Hasta ahora hemos entrenado nuestros propios modelos con `pacientes.csv`. Aquí cambiamos de rol por completo: **no entrenamos nada**. Vamos a **usar** tres modelos que otros ya entrenaron —descargándolos de **Hugging Face**, "el GitHub de los modelos"— y verás que "usar" cabe, literalmente, en tres líneas de código.

> ⚠️ Todos los datos de partida son **sintéticos** (generados por código, semilla fija). **No representan pacientes reales.** La primera celda los crea sola: no hay que descargar nada. Los **modelos**, en cambio, son reales y vienen de internet: la primera vez que se usan, se descargan (puede tardar y necesita conexión). Si tu Colab no tiene red o falta algún paquete, cada ejemplo está protegido para que el cuaderno **siga adelante sin romperse** y te explique qué ha pasado.

**Qué haremos, en clave clínica:**
1. Entender qué es una **`pipeline()`** y por qué es "la forma más simple de usar un modelo ya entrenado".
2. **Extraer entidades biomédicas** (enfermedades, síntomas, fármacos) de una frase clínica en **inglés**, con `d4data/biomedical-ner-all`.
3. Asomarnos a un modelo **fundacional biomédico-clínico en español**, `PlanTL-GOB-ES/roberta-base-biomedical-clinical-es`, viendo cómo "rellena huecos" en una frase clínica tomada de `notas_clinicas.csv`.
4. **Mencionar** cómo se clasificaría una radiografía de tórax con un modelo ya afinado del Hub, `dima806/chest_xray_pneumonia_detection`.
5. Cerrar con el **cuestionario clínico** que hay que hacerse antes de fiarse de cualquiera de estos modelos.

[Abrir en Colab](PENDIENTE_DRIVE)


## 0. Preparamos los datos (esta celda se ejecuta sola)

La primera celda de cada cuaderno del curso **genera los ficheros sintéticos** en la carpeta de trabajo, entre ellos `notas_clinicas.csv`, que usaremos como fuente de frases clínicas de ejemplo. Ejecútala una vez (▶ o *Mayús+Enter*) y sigue hacia abajo.


In [ ]:
# === Generación de los datos sintéticos del curso (se ejecuta sola) ===
# TODOS LOS DATOS SON SINTÉTICOS. No representan pacientes reales. Semilla fija -> reproducible.
import os, numpy as np, pandas as pd

def generar_datos_clinicos(carpeta="."):
    """Crea los CSV del curso si no están ya en la carpeta. Devuelve la ruta."""
    if os.path.exists(os.path.join(carpeta, "pacientes.csv")):
        return carpeta
    RNG = np.random.default_rng(2026)   # misma semilla en todo el curso

    # --- Centros ---
    AREAS = ["Valencia","Alicante","Castellón","Madrid","Barcelona","Sevilla","Bilbao","Zaragoza"]
    nc = 24
    tipo = RNG.choice(["hospital","centro de salud"], nc, p=[0.35,0.65])
    camas = np.where(tipo=="hospital", RNG.integers(120,900,nc), RNG.integers(0,25,nc))
    nserv = np.where(tipo=="hospital", RNG.integers(12,40,nc), RNG.integers(3,10,nc))
    centros = pd.DataFrame({"centro_id":[f"C{i:03d}" for i in range(1,nc+1)],"tipo":tipo,
        "area":RNG.choice(AREAS,nc),"camas":camas,"n_servicios":nserv,
        "urgencias_dia_media":np.where(tipo=="hospital",RNG.integers(80,320,nc),RNG.integers(5,40,nc)),
        "ratio_mayores65":RNG.uniform(0.12,0.34,nc).round(3)})
    centros.to_csv(os.path.join(carpeta,"centros.csv"), index=False)

    # --- Pacientes (tabla central) ---
    N = 20000
    sexo = RNG.choice(["M","F"], N, p=[0.49,0.51])
    edad = np.clip(np.where(RNG.random(N)<0.6, RNG.normal(58,15,N), RNG.normal(40,12,N)),18,95).round().astype(int)
    p_act = np.clip(0.28-0.0016*(edad-18),0.06,0.28); p_ex = np.clip(0.10+0.004*(edad-18),0.10,0.40); p_nu = 1-p_act-p_ex
    tabaquismo = np.array([RNG.choice(["nunca","ex","activo"],p=[a,b,c]) for a,b,c in zip(p_nu,p_ex,p_act)])
    actividad = RNG.choice(["baja","media","alta"], N, p=[0.4,0.4,0.2])
    antec = RNG.binomial(1,0.22,N)
    actn = pd.Series(actividad).map({"baja":1.5,"media":0.0,"alta":-1.3}).values
    imc = np.clip(RNG.normal(26.5,4.2,N)+0.03*(edad-50)+actn+RNG.normal(0,1.0,N),15,48).round(1)
    fa = (tabaquismo=="activo").astype(float); fe = (tabaquismo=="ex").astype(float)
    ta_s = np.clip(RNG.normal(120,12,N)+0.45*(edad-45)+0.7*(imc-25)+6*fa+RNG.normal(0,6,N),85,220).round().astype(int)
    ta_d = np.clip(0.55*ta_s+RNG.normal(20,6,N),50,130).round().astype(int)
    glu = np.clip(RNG.normal(100,12,N)+1.0*(imc-25)+0.3*(edad-45)+9*antec+RNG.normal(0,7,N),60,320).round(1)
    diabetes = (glu>126).astype(int)
    col = np.clip(RNG.normal(195,30,N)+0.4*(edad-45)+1.1*(imc-25)+RNG.normal(0,15,N),110,380).round(1)
    hdl = np.clip(RNG.normal(55,12,N)-0.25*(imc-25)+6*(sexo=="F")-5*fa-3*actn+RNG.normal(0,6,N),20,110).round(1)
    z = (-3.1+0.062*(edad-50)+0.019*(ta_s-120)+0.008*(col-190)-0.028*(hdl-55)+0.055*(imc-25)
         +0.85*fa+0.30*fe+0.60*diabetes+0.45*antec+0.55*(sexo=="M")
         +0.012*np.maximum(edad-65,0)*fa+RNG.normal(0,0.35,N))
    p = 1/(1+np.exp(-z))
    riesgo = (100*p).round(1); evento = RNG.binomial(1,p)
    pacientes = pd.DataFrame({"paciente_id":[f"P{i:05d}" for i in range(1,N+1)],"edad":edad,"sexo":sexo,
        "imc":imc,"ta_sistolica":ta_s,"ta_diastolica":ta_d,"glucemia":glu,"colesterol_total":col,"hdl":hdl,
        "tabaquismo":tabaquismo,"actividad_fisica":actividad,"antecedentes_familiares":antec,
        "diabetes":diabetes,"riesgo_cv_10a":riesgo,"evento_cv":evento})
    pacientes.to_csv(os.path.join(carpeta,"pacientes.csv"), index=False)

    # --- Versión "sucia" para EDA ---
    d = pacientes.copy()
    d["sexo"] = d["sexo"].map(lambda s: RNG.choice({"M":["M","m","Masculino","H","Hombre"],"F":["F","f","Femenino","Mujer"]}[s]))
    idx = RNG.choice(d.index,int(N*0.06),replace=False)
    d["glucemia"] = d["glucemia"].astype(object)   # permite mezclar texto (mmol/L con coma) y números
    d.loc[idx,"glucemia"] = [str(round(v/18.0,1)).replace(".",",") for v in d.loc[idx,"glucemia"]]
    prob_na = np.where(pacientes["edad"]<40,0.12,0.04); mask = RNG.random(N)<prob_na
    d.loc[mask,"hdl"] = np.nan
    d.loc[RNG.choice(d.index,int(N*0.05),replace=False),"colesterol_total"] = np.nan
    d.loc[RNG.choice(d.index,25,replace=False),"edad"] = RNG.integers(150,260,25)
    d.loc[RNG.choice(d.index,20,replace=False),"ta_sistolica"] = -RNG.integers(1,90,20)
    d.loc[RNG.choice(d.index,15,replace=False),"imc"] = RNG.uniform(80,200,15).round(1)
    d["colesterol_total"] = d["colesterol_total"].astype(object); d["ta_diastolica"] = d["ta_diastolica"].astype(object)
    d.loc[RNG.choice(d.index,30,replace=False),"colesterol_total"] = "desconocido"
    d.loc[RNG.choice(d.index,20,replace=False),"ta_diastolica"] = "N/D"
    d["tabaquismo"] = d["tabaquismo"].map(lambda s: RNG.choice({"nunca":["nunca","No fumador","NUNCA"],
        "ex":["ex","exfumador","Ex-fumador"],"activo":["activo","Fumador","SI"]}[s]))
    d = pd.concat([d, d.sample(400,random_state=3)], ignore_index=True)
    idx = RNG.choice(d.index,200,replace=False); d.loc[idx,"paciente_id"] = d.loc[idx,"paciente_id"].map(lambda s:" "+s+" ")
    d = d.sample(frac=1,random_state=11).reset_index(drop=True)
    d.to_csv(os.path.join(carpeta,"pacientes_sucio.csv"), index=False)

    # --- Urgencias (serie temporal) ---
    ND = 730; fechas = pd.date_range("2024-01-01", periods=ND, freq="D")
    doy = fechas.dayofyear.values; dow = fechas.dayofweek.values
    fest_set = {(1,1),(1,6),(5,1),(8,15),(10,12),(11,1),(12,6),(12,8),(12,25)}
    festivo = np.array([1 if (f.month,f.day) in fest_set else 0 for f in fechas])
    gripe = np.array([1 if f.month in (12,1,2) else 0 for f in fechas])
    temp = (15+10*np.sin(2*np.pi*(doy-110)/365)+RNG.normal(0,2.5,ND)).round(1)
    est = 1+0.14*np.sin(2*np.pi*(doy-20)/365)
    sem = np.where(dow==0,1.18,np.where(dow>=5,1.10,1.0))
    mu = 120.0*est*sem*(1+0.22*gripe)*(1+0.15*festivo)
    ingresos = RNG.poisson(np.maximum(mu,1)).astype(int)
    pd.DataFrame({"fecha":fechas.strftime("%Y-%m-%d"),"ingresos":ingresos,"festivo":festivo,
        "temporada_gripe":gripe,"temperatura":temp}).to_csv(os.path.join(carpeta,"urgencias_diarias.csv"), index=False)

    # --- Notas clínicas (texto) ---
    plant = {"cardiología":["dolor torácico opresivo de {t} de evolución, irradiado a brazo izquierdo",
        "disnea de esfuerzo progresiva y palpitaciones, antecedente de hipertensión",
        "control de anticoagulación, fibrilación auricular conocida, sin dolor actual",
        "edemas en miembros inferiores y ortopnea, sospecha de insuficiencia cardiaca"],
      "respiratorio":["tos productiva de {t}, fiebre y disnea, auscultación con crepitantes",
        "sibilancias y opresión torácica, antecedente de asma, mala respuesta a inhalador",
        "disnea súbita y dolor pleurítico, saturación disminuida",
        "tos seca persistente y febrícula, contacto con caso respiratorio"],
      "digestivo":["dolor abdominal en fosa iliaca derecha de {t}, náuseas y febrícula",
        "epigastralgia y pirosis, relación con las comidas, sin signos de alarma",
        "diarrea de {t} sin productos patológicos, buen estado general",
        "ictericia y coluria, dolor en hipocondrio derecho"],
      "neurología":["cefalea intensa de inicio súbito, la peor de su vida, con fotofobia",
        "focalidad neurológica de {t}, debilidad en hemicuerpo y disartria",
        "mareo con giro de objetos y vómitos, sin cortejo vegetativo",
        "crisis comicial presenciada, recuperación progresiva del nivel de conciencia"],
      "traumatología":["caída casual con dolor e impotencia funcional en muñeca, deformidad",
        "lumbalgia mecánica de {t} tras esfuerzo, sin déficit neurológico",
        "esguince de tobillo con edema y dificultad para la deambulación",
        "gonalgia y derrame articular tras traumatismo deportivo"]}
    prio = {"cardiología":[0.35,0.45,0.20],"respiratorio":[0.30,0.45,0.25],"digestivo":[0.20,0.50,0.30],
        "neurología":[0.45,0.35,0.20],"traumatología":[0.12,0.48,0.40]}
    tiempos = ["horas","un día","dos días","una semana","varios días"]; esp_list = list(plant.keys()); rows=[]
    for _ in range(3000):
        e = RNG.choice(esp_list); txt = RNG.choice(plant[e]).replace("{t}",RNG.choice(tiempos))
        rows.append((txt,e,RNG.choice(["alta","media","baja"],p=prio[e]),RNG.choice(centros["centro_id"].values)))
    pd.DataFrame(rows,columns=["texto","especialidad","prioridad","centro_id"]).to_csv(os.path.join(carpeta,"notas_clinicas.csv"),index=False)

    # --- Wearable (señal) ---
    rows=[]
    for s in range(1,201):
        fcb=RNG.normal(66,6); pb=RNG.normal(7500,2500); sb=RNG.normal(7.0,0.8); anom=RNG.random()<0.05
        for dia in range(1,31):
            fc=fcb+RNG.normal(0,3); pasos=max(0,pb+RNG.normal(0,1500)); sue=float(np.clip(sb+RNG.normal(0,0.6),3,11))
            if anom and dia in (14,15,16): fc+=RNG.uniform(18,30); sue-=RNG.uniform(1.5,3)
            rows.append((f"S{s:03d}",dia,round(fc,1),int(pasos),round(sue,1)))
    pd.DataFrame(rows,columns=["sujeto_id","dia","fc_reposo","pasos","horas_sueno"]).to_csv(os.path.join(carpeta,"wearable.csv"),index=False)
    return carpeta

generar_datos_clinicos(".")
print("Datos sintéticos listos en la carpeta actual. (Recuerda: son datos SINTÉTICOS, no reales.)")

### 0.1 Instalamos `transformers` (si hiciera falta)

En Google Colab, la librería `transformers` de Hugging Face suele venir **ya instalada**. Por si acaso, esta celda comprueba si está disponible y, si no, intenta instalarla. Está envuelta en `try/except` para que, aunque no haya conexión a internet, el cuaderno **no se detenga**: simplemente te avisará.


In [ ]:
try:
    import transformers
    print(f"transformers ya está disponible (versión {transformers.__version__}).")
except Exception as e:
    print("transformers no estaba disponible; probamos a instalarlo con pip...")
    try:
        import subprocess
        subprocess.run(["pip", "install", "-q", "transformers"], check=True)
        import transformers
        print(f"Instalado. Versión {transformers.__version__}.")
    except Exception as e2:
        print("No se ha podido instalar transformers en este entorno.")
        print("Motivo:", repr(e2))
        print("En Colab, ejecuta en una celda: !pip install transformers")

**Qué acabamos de hacer:** nada clínico todavía, solo asegurarnos de que la "caja de herramientas" (`transformers`) está lista. A partir de aquí, cada ejemplo con un modelo del Hub irá también protegido con su propio `try/except`: si algo falla —sin red, sin GPU, un nombre de modelo que cambia— el cuaderno te lo dirá con claridad y **seguirá corriendo**, en lugar de detenerse con un error críptico.


## 1. `pipeline()`: usar un modelo entrenado en tres líneas

Piensa en `pipeline()` como un **enchufe universal**. Detrás de cada modelo del Hub hay pasos técnicos —preparar el texto o la imagen en el formato que el modelo espera, ejecutar la red, ordenar la salida en algo legible— que normalmente exigirían bastante código. La función `pipeline()` **empaqueta todos esos pasos** detrás de una sola llamada: le dices **qué tarea** quieres (por ejemplo, `"token-classification"` o `"image-classification"`) y, opcionalmente, **qué modelo** del Hub usar; ella descarga el modelo la primera vez, lo prepara y te devuelve un objeto al que solo tienes que **pasarle tu dato**.

El patrón se repite siempre igual:

```python
from transformers import pipeline

tarea = pipeline("nombre-de-la-tarea", model="autor/nombre-del-modelo")
resultado = tarea(mi_dato)
```

Tres líneas: importar, crear la *pipeline*, usarla. Eso es "usar un modelo fundacional" en la práctica. Lo que cambia de un ejemplo a otro es solo el **nombre de la tarea** y el **nombre del modelo** — el patrón de fondo es idéntico.


{% hint style="warning" %}
**⚠️ Aviso, antes de empezar: fácil de ejecutar no es lo mismo que válido para tus pacientes**

Que un modelo responda en tres líneas no dice nada sobre si es fiable **para tu población**. Cada modelo se entrenó con unos datos, un idioma y un contexto concretos, que pueden no parecerse a los tuyos. Trátalos como lo que son en este cuaderno: una **demostración de cómo se usan**, no una herramienta clínica lista para producción. Volvemos sobre esto, con detalle, en la sección 5.
{% endhint %}


## 2. Extraer entidades biomédicas de una frase clínica (inglés)

Nuestro primer modelo real es `d4data/biomedical-ner-all`, entrenado para la tarea `token-classification`: dado un texto libre, **localiza y etiqueta** menciones de enfermedades, síntomas, fármacos y otras entidades biomédicas. Es una de las tareas más útiles de PLN clínico: convertir una frase en información **estructurada** (una lista de entidades con su tipo), en lugar de dejarla como texto plano.

Este modelo concreto trabaja en **inglés**. Usaremos una frase clínica de ejemplo típica en inglés — piénsala como una nota que podría llegar de un informe en ese idioma, o como la traducción de una nota como las de nuestro `notas_clinicas.csv` (que están en español; el siguiente bloque usa precisamente ese fichero con un modelo que sí entiende español).


In [ ]:
from transformers import pipeline

frase_en = "Patient with type 2 diabetes and hypertension, started on metformin."

try:
    ner = pipeline("token-classification",
                    model="d4data/biomedical-ner-all",
                    aggregation_strategy="simple")

    print("Frase clínica de ejemplo (inglés):")
    print(" ", frase_en)
    print()
    print("Entidades detectadas:")
    for entidad in ner(frase_en):
        print(f"  {entidad['word']:20s} -> {entidad['entity_group']}  (confianza {entidad['score']:.2f})")

except Exception as e:
    print("No se ha podido ejecutar el modelo de NER biomédico en este entorno.")
    print("Motivo:", repr(e))
    print("Causas típicas: sin conexión a internet, o el modelo aún se está descargando la primera vez.")
    print("En Colab con red, este mismo código detecta 'diabetes' y 'hypertension' como Disease_disorder,")
    print("y 'metformin' como Medication.")

**Lo que vemos (si hay red):** el modelo separa la frase en fragmentos y etiqueta cada uno con su tipo de entidad biomédica: **diabetes** e **hypertension** como enfermedad/trastorno, **metformin** como medicación. No ha "entendido" la frase como lo haría un clínico, pero ha aprendido —a partir de muchísimo texto biomédico— a **reconocer patrones** que corresponden a esas categorías. Convertir texto libre en una lista de entidades es exactamente el tipo de tarea que alimenta después una historia clínica estructurada o una búsqueda por síntomas.

**Si ha fallado (sin red):** no pasa nada — el mensaje del `except` te explica qué habría pasado y por qué. La lección no cambia: en tres líneas, sin entrenar nada, un texto libre se convierte en información estructurada.


## 3. Un modelo fundacional clínico en español

Para el mundo hispanohablante existe `PlanTL-GOB-ES/roberta-base-biomedical-clinical-es`: un modelo **RoBERTa** preentrenado sobre texto biomédico y clínico **en español**, desarrollado en el marco del Plan de Tecnologías del Lenguaje. Es un modelo fundacional "de base": no viene ya resuelto para una tarea concreta (no clasifica ni extrae nada por sí solo), sino que **rellena huecos** — le das una frase con una palabra oculta y te propone qué palabra encaja ahí. Esa capacidad de "adivinar" revela cuánto lenguaje clínico en español ha aprendido, y es la **materia prima** que después se afina para tareas como el NER en español.

Vamos a tomar una frase real de nuestro `notas_clinicas.csv` (recuerda: **sintética**), ocultar una palabra clínicamente relevante con `<mask>`, y ver qué propone el modelo.


In [ ]:
import pandas as pd

notas = pd.read_csv("notas_clinicas.csv")
print("Primeras notas clínicas sintéticas disponibles:")
notas[["texto", "especialidad"]].head(3)

**Lo que vemos:** son frases cortas de estilo clínico, con su especialidad asociada. Vamos a construir nuestra frase con `<mask>` a partir de una nota real de cardiología del propio fichero — la misma idea que el ejemplo clásico "El paciente presenta dolor `<mask>` de esfuerzo".


In [ ]:
# Tomamos una nota de cardiología del propio dataset sintético y ocultamos una palabra clínica
nota_cardio = notas[notas["especialidad"] == "cardiología"]["texto"].iloc[0]
print("Nota original:")
print(" ", nota_cardio)

frase_es = "El paciente presenta dolor <mask> de esfuerzo."
print()
print("Frase con hueco que le pasaremos al modelo:")
print(" ", frase_es)

In [ ]:
from transformers import pipeline

try:
    base_es = pipeline("fill-mask",
                        model="PlanTL-GOB-ES/roberta-base-biomedical-clinical-es")

    propuestas = base_es(frase_es)
    print("Propuestas del modelo para <mask>:")
    for p in propuestas:
        print(f"  '{p['token_str'].strip():15s}'  (probabilidad {p['score']:.2f})  -> {p['sequence']}")

except Exception as e:
    print("No se ha podido ejecutar el modelo biomédico-clínico en español en este entorno.")
    print("Motivo:", repr(e))
    print("Causas típicas: sin conexión a internet, o el modelo (varios cientos de MB) aún se está descargando.")
    print("En Colab con red, este mismo código propone términos como 'torácico' o 'precordial' para <mask>,")
    print("mostrando que el modelo ha aprendido vocabulario clínico en español de verdad.")

**Lo que significa (si hay red):** el modelo no "sabe cardiología" en el sentido clínico, pero ha visto tanto texto biomédico en español durante su entrenamiento que **propone términos clínicamente plausibles** para el hueco — típicamente algo como "torácico" o "precordial", justo el vocabulario que un profesional esperaría ahí. Este es el puente de la unidad: existen modelos fundacionales del lenguaje clínico en español, ya entrenados y disponibles en el Hub, listos para que **se adapten** después a una tarea concreta (extraer entidades, clasificar notas por prioridad...) con relativamente pocos datos propios — en lugar de entrenar desde cero un modelo del español clínico, algo fuera del alcance de cualquier equipo asistencial.

**Nota importante:** este bloque es un **puente** hacia el trabajo con texto clínico, no una clase de procesamiento de lenguaje natural. Nos basta con la idea grande: el modelo existe, está en español, y se puede tomar del Hub y adaptar.


## 4. Imagen médica: un ViT ya afinado para radiografías de tórax

El tercer ejemplo es de **imagen**. En el Hub existe `dima806/chest_xray_pneumonia_detection`: un **Vision Transformer (ViT)** ya afinado para clasificar radiografías de tórax en *neumonía* / *normal*. El patrón de uso es idéntico a los anteriores, cambiando solo la tarea a `"image-classification"`:

```python
clasificador = pipeline("image-classification",
                         model="dima806/chest_xray_pneumonia_detection")
clasificador("radiografia_ejemplo.png")
# -> [{'label': 'PNEUMONIA', 'score': 0.97}, {'label': 'NORMAL', 'score': 0.03}]
```

En este cuaderno **no vamos a ejecutar** este clasificador con una imagen real: hacerlo bien exigiría descargar y guardar una radiografía pública de ejemplo, y el objetivo aquí es que reconozcas el **patrón** — el mismo `pipeline()`, la misma estructura en tres líneas — más que repetir la descarga de una imagen. Sí vamos a comprobar que la *pipeline* se puede **crear** (que el modelo existe y es accesible), sin clasificar nada todavía.


In [ ]:
from transformers import pipeline

try:
    clasificador_rx = pipeline("image-classification",
                                model="dima806/chest_xray_pneumonia_detection")
    print("Pipeline de clasificación de radiografías creada correctamente.")
    print("Modelo cargado:", clasificador_rx.model.config.name_or_path)
    print("Etiquetas que reconoce:", list(clasificador_rx.model.config.id2label.values()))
    print()
    print("(No clasificamos ninguna imagen en este cuaderno: solo confirmamos que el modelo")
    print(" se puede descargar y queda listo para usar. Para clasificar de verdad, se le pasaría")
    print(" la ruta de una imagen pública de radiografía, nunca la de un paciente real.)")

except Exception as e:
    print("No se ha podido cargar el clasificador de radiografías en este entorno.")
    print("Motivo:", repr(e))
    print("Causas típicas: sin conexión a internet, o el modelo aún se está descargando la primera vez.")
    print("En Colab con red, este mismo código carga un Vision Transformer (ViT) ya afinado")
    print("para distinguir 'PNEUMONIA' de 'NORMAL' en radiografías de tórax.")

**Lo que importa aquí no es la métrica, es el patrón.** Con exactamente la misma estructura de tres líneas — importar, crear la *pipeline*, pasarle el dato — hemos usado un modelo de **texto en inglés** (bloque 2), un modelo **fundacional en español** (bloque 3) y ahora confirmamos que un modelo de **imagen médica** se activa igual. Cambia la tarea y el nombre del modelo; el patrón de uso es el mismo siempre. Esa es, probablemente, la idea más importante de todo el cuaderno.


## 5. Antes de fiarte: el cuestionario clínico

Los tres modelos han respondido en segundos y sin entrenar nada. Precisamente por eso, la responsabilidad de decidir si sirven para tu contexto es **tuya**, no del modelo. Dos avisos, por orden de gravedad.


{% hint style="danger" %}
**⚠️ Privacidad: nunca envíes datos reales de pacientes a un modelo del Hub sin garantías**

Aunque en este cuaderno los modelos de Hugging Face se ejecutan **en tu propio entorno** (Colab, un servidor del hospital) — no viajan a un tercero como en la Vía B (APIs, que veremos en el siguiente cuaderno) —, sigue existiendo un riesgo si el **modelo en sí** no está evaluado ni desplegado con las garantías adecuadas: usarlo para tomar decisiones sobre una persona real sin validación es tan arriesgado como enviar el dato a una API pública. La regla de este curso: en los ejercicios, **solo texto e imágenes de ejemplo, sintéticos o públicos**; nunca una nota o una radiografía de un paciente real.
{% endhint %}


{% hint style="warning" %}
**⚠️ Validación: "funciona en el Hub" no es "funciona en tu hospital"**

Cada uno de estos tres modelos se entrenó con unos datos, una población y un idioma concretos. El NER biomédico solo entiende **inglés**. El modelo clínico en español se entrenó con un tipo de texto que puede no coincidir con el estilo de tus informes. El clasificador de radiografías aprendió de un conjunto de imágenes de un origen y un protocolo que pueden no parecerse a los tuyos — y, como vimos en la unidad anterior, podría haber aprendido un **atajo espurio** (fijarse en el aparato o el protocolo, no en la enfermedad). Antes de fiarte de cualquiera de estos modelos para algo real: ¿con qué datos se entrenó?, ¿está **validado en tu población**?, ¿qué tasa de error tiene por subgrupos? Un modelo público es un punto de partida para investigar, no una herramienta clínica lista para usar.
{% endhint %}


{% hint style="success" %}
**💡 Idea clave**

Las tres líneas de `pipeline()` son la parte **fácil**. La parte **difícil** —y la que de verdad importa en salud— es decidir si ese modelo, entrenado por otros con datos que no conoces del todo, tiene sentido para tus pacientes. Cuanto más fácil es ejecutar un modelo ajeno, mayor es la responsabilidad de **no hacerlo a ciegas**.
{% endhint %}


## 6. Qué hemos aprendido

- **`pipeline()` es el patrón universal para "usar" un modelo del Hub**: importar, crear la *pipeline* con la tarea y el modelo, pasarle el dato. Tres líneas, siempre igual, cambie lo que cambie por dentro.
- **Probamos tres tipos de modelo, sin entrenar ninguno**: un extractor de entidades biomédicas en **inglés** (`d4data/biomedical-ner-all`), un modelo fundacional **clínico en español** que rellena huecos (`PlanTL-GOB-ES/roberta-base-biomedical-clinical-es`) y la confirmación de que un **clasificador de radiografías** (`dima806/chest_xray_pneumonia_detection`) se activa con el mismo patrón.
- **El modelo en español es un puente, no una clase de PLN**: existen modelos fundacionales del lenguaje clínico en español, ya entrenados, listos para adaptarse a tareas concretas con pocos datos propios.
- **Fácil de ejecutar no es lo mismo que válido para tus pacientes.** Antes de fiarte de un modelo del Hub: ¿en qué datos se entrenó?, ¿está validado en tu población?, ¿qué idioma y qué contexto asume?
- **La responsabilidad crece con la facilidad.** Que "usar" sea trivial no reduce tu criterio clínico: lo hace más necesario que nunca.

**Puente al siguiente cuaderno:** los tres modelos de hoy se **descargan** y corren en tu propio entorno — la Vía A del mapa de la unidad. Pero no todos los modelos potentes se pueden descargar: los grandes modelos conversacionales (Claude, GPT, Gemini, Qwen) son, en su mayoría, propietarios y se usan **a distancia**, por API. Esa es la Vía B, y es exactamente lo que abrimos en `U09b_APIs_OpenRouter.ipynb`.


## 7. Y esto, ¿cómo se lo pido a un asistente de IA?

Recuerda el principio del curso: **tú pones el criterio clínico, la IA escribe el código.** Un buen encargo para reproducir (o ampliar) este cuaderno sería:

> *"En español y por celdas, con la librería `transformers` de Hugging Face: (1) explícame en dos frases qué es una `pipeline` y por qué me deja usar un modelo ya entrenado sin entrenar nada. (2) Crea una pipeline de `token-classification` con el modelo `d4data/biomedical-ner-all` y extráeme las entidades de una frase clínica en inglés que yo te dé. (3) Crea una pipeline de `fill-mask` con `PlanTL-GOB-ES/roberta-base-biomedical-clinical-es` y pruébala con una frase clínica en español con un hueco `<mask>`. Envuelve cada llamada al modelo en `try/except` con un mensaje claro, para que el cuaderno no se rompa si falta conexión. (4) Al final, enumérame qué debería comprobar antes de fiarme de estos modelos en mi hospital: población de entrenamiento, validación externa, sesgos, idioma, versión del modelo."*

Fíjate en el punto (4): no le pides solo que ejecute, le pides el **cuestionario clínico**. Ese es el criterio que distingue a un profesional de un usuario ingenuo de la IA — y es exactamente lo que no se automatiza.
